# ArcGIS service REST client (`geoREST.RESTesri.services`) — usage examples

Walks through every public function in `src/geoREST/RESTesri/services.py` against live ArcGIS REST services (Image Services, Feature Services, mosaic layers):

- `getImageServiceTileUrl` — build a `{z}/{y}/{x}` tile URL template
- `queryFeatureServiceCount` — pre-flight feature count for a where/spatial filter
- `queryFeatureService` — fetch features as GeoJSON
- `getLayerInfo` — layer-level metadata (fields, geometry type, capabilities)
- `exportImage` — export a rendered image from an Image Service
- `getSupportedOperations` — list every REST operation a service actually exposes
- `computeStatisticsHistograms` — per-band pixel statistics + histogram over an area
- `identifyPixelValue` — identify the pixel value at a point
- `getSamples` — sample pixel values at multiple points
- `queryBoundary` — the true (non-rectangular) coverage boundary of an Image Service

Two live test services are used throughout:

- **Image Service** — [LCMS Annual Change (CONUS)](https://imagery.geoplatform.gov/iipp/rest/services/Vegetation/USFS_EDW_LCMS_AnnualChange_CONUS/ImageServer), a USFS mosaic dataset where each raster in the catalog is one calendar year of the Landscape Change Monitoring System's annual change classification. Filtering to a single year is done with a mosaic rule (`{"where": "year=<yyyy>"}`) rather than a URL path, since the "layers" here are catalog rows, not sub-services.
- **Cached Map Service** — Esri's public [`World_Imagery`](https://services.arcgisonline.com/arcgis/rest/services/World_Imagery/MapServer) basemap, used only for `getImageServiceTileUrl`: it is *tile-cached*, which the LCMS Image Service is not.
- **Feature Service** — Esri's public [`Earthquakes_Since1970`](https://sampleserver6.arcgisonline.com/arcgis/rest/services/Earthquakes_Since1970/FeatureServer/0) sample service: worldwide earthquake point records with a `magnitude` field, good for both attribute and spatial (bbox) filtering.

## Setup

This notebook needs `geoREST` installed:

```
pip install georest
```

or, from a clone of this repository, `pip install -e .` from the repo root.


In [ ]:
from geoREST.RESTesri.services import (
    getImageServiceTileUrl,
    queryFeatureServiceCount,
    queryFeatureService,
    getLayerInfo,
    exportImage,
    getSupportedOperations,
    computeStatisticsHistograms,
    identifyPixelValue,
    getSamples,
    queryBoundary,
)

In [ ]:
IMG = "https://imagery.geoplatform.gov/iipp/rest/services/Vegetation/USFS_EDW_LCMS_AnnualChange_CONUS/ImageServer"
FEATURES = "https://sampleserver6.arcgisonline.com/arcgis/rest/services/Earthquakes_Since1970/FeatureServer/0"
# A tile-cached service, for the tile-URL example below. LCMS is not cached.
TILED = "https://services.arcgisonline.com/arcgis/rest/services/World_Imagery/MapServer"

# A small AOI near the 2020 Castle Fire (Sequoia National Forest, CA) used for
# the Image Service examples below.
AOI_BBOX = "-118.65,36.05,-118.45,36.25"
YEAR_2020 = {"where": "year=2020"}  # mosaic rule: restrict LCMS to a single year

## 1. `getImageServiceTileUrl` — build a tile URL template

ArcGIS tile URLs use `{z}/{y}/{x}` order (y before x), not the XYZ standard `{z}/{x}/{y}`.

This is **pure string construction** — no request is made and nothing is validated. In particular it cannot tell whether the service is actually tiled: only *cached* services serve `/tile`. So we point it at `TILED` (a cached Map Service) rather than at the LCMS Image Service used everywhere else in this notebook, which has no tile cache at all.

In [ ]:
getImageServiceTileUrl(TILED)

Building a template against an **uncached** service silently succeeds — the string is well-formed, but every tile it points at answers HTTP 404. Check for `tileInfo` in the service's `?f=json` before relying on tiles; render an uncached Image Service with `exportImage` (section 5) instead.

In [ ]:
from geoREST.RESTesri._http import fetch_bytes, fetch_json

print("World_Imagery tiled?", "tileInfo" in fetch_json(TILED, {"f": "json"}))
print("LCMS tiled?        ", "tileInfo" in fetch_json(IMG, {"f": "json"}))

raw, ctype = fetch_bytes(getImageServiceTileUrl(TILED).format(z=6, y=25, x=12))
print(f"cached tile  -> {len(raw):,} bytes of {ctype}")

try:
    fetch_bytes(getImageServiceTileUrl(IMG).format(z=6, y=25, x=12))
except RuntimeError as exc:
    print("uncached tile ->", exc)

## 2. `queryFeatureServiceCount` — pre-flight feature count

A cheap `returnCountOnly=true` request — useful to check how big a result set would be before fetching geometry with `queryFeatureService`.

In [ ]:
queryFeatureServiceCount(FEATURES)  # every earthquake record in the service

In [ ]:
queryFeatureServiceCount(FEATURES, where="magnitude>=7")

## 3. `queryFeatureService` — fetch features as GeoJSON

Runs the same count pre-flight as above, then fetches geometry + attributes as a GeoJSON `FeatureCollection`. `where` filters by attribute; `geometry`/`geometry_type` add a spatial filter (bbox string, GeoJSON geometry dict, or Esri JSON all work).

In [ ]:
biggest = queryFeatureService(FEATURES, where="magnitude>=8", out_fields="name,magnitude,date_")
sorted((f["properties"] for f in biggest["features"]), key=lambda p: -p["magnitude"])[:5]

A bbox string as the spatial filter — every California-area quake of magnitude 6+:

In [ ]:
ca_quakes = queryFeatureService(
    FEATURES,
    geometry="-125,32,-114,42", geometry_type="esriGeometryEnvelope",
    where="magnitude>=6", out_fields="name,magnitude,date_",
)
print(len(ca_quakes["features"]), "California-area quakes with magnitude >= 6")
[f["properties"] for f in ca_quakes["features"]]

`max_features` guards against accidentally pulling a huge result set — the count pre-flight catches it before any geometry is fetched:

In [ ]:
try:
    queryFeatureService(FEATURES, where="magnitude>=5", max_features=50)
except ValueError as e:
    print(e)

`queryFeatureService` also works directly against an Image Service's mosaic catalog (each row = one contributing raster) — here, the single LCMS raster for 2020. Its native `/query` rejects `f=geojson` outright, so this transparently falls back to `f=json` and converts the Esri JSON to GeoJSON client-side.

In [ ]:
mosaic_rows = queryFeatureService(IMG, where="year=2020", out_fields="name,year,category")
mosaic_rows["features"][0]["properties"], mosaic_rows["features"][0]["geometry"]["type"]

## 4. `getLayerInfo` — layer-level metadata

Fields, geometry type, and capabilities. Works for a FeatureServer sub-layer or an Image Service's mosaic footprint layer alike.

In [ ]:
info = getLayerInfo(FEATURES)
print(info["name"], "-", info["geometryType"])
print("capabilities:", info["capabilities"])
[f["name"] for f in info["fields"]]

In [ ]:
img_info = getLayerInfo(IMG)
print(img_info["name"])
[f["name"] for f in img_info["fields"]]

## 5. `exportImage` — export a rendered image

Exports a rendered PNG for the 2020 LCMS Change raster over the AOI, using the service's built-in `Annual_Change` raster function (colormap) as the `rendering_rule` and the `year=2020` mosaic rule to pick that single year out of the catalog.

In [ ]:
from IPython.display import Image

png_path = exportImage(
    IMG,
    bbox=AOI_BBOX,
    size=(400, 400),
    image_format="png",
    mosaic_rule=YEAR_2020,
    rendering_rule={"rasterFunction": "Annual_Change"},
    out_path="lcms_2020_change.png",
)
Image(png_path)

## 6. `getSupportedOperations` — every REST operation a service exposes

The `?f=json` `capabilities` string is coarse and unreliable (some operations, like `computeStatisticsHistograms` below, have no corresponding capability flag at all). This parses the authoritative list off the service's HTML browse page instead.

In [ ]:
for op in getSupportedOperations(FEATURES):
    print(op["name"], "->", op["operation"])

In [ ]:
for op in getSupportedOperations(IMG):
    print(op["name"], "->", op["operation"])

## 7. `computeStatisticsHistograms` — per-band pixel statistics + histogram

Computed over the AOI at native resolution by default. LCMS Change values are class codes (this is a categorical raster), so `histogram.counts` — pixel counts per class value — is more informative here than `mean`/`stddev`.

In [ ]:
stats = computeStatisticsHistograms(IMG, geometry=AOI_BBOX, geometry_type="esriGeometryEnvelope", mosaic_rule=YEAR_2020)
stats

`pixel_size` computes at a coarser/finer resolution — but it's interpreted in the units of *its own* `spatialReference`, not the raster's native units. Requesting `0.001` degrees (~100 m) works fine; requesting `90` *degrees* (rather than 90 m) asks for an absurdly coarse resolution and the service silently returns nothing rather than erroring:

In [ ]:
computeStatisticsHistograms(
    IMG, geometry=AOI_BBOX, geometry_type="esriGeometryEnvelope",
    mosaic_rule=YEAR_2020,
    pixel_size={"x": 0.001, "y": 0.001, "spatialReference": {"wkid": 4326}},
)

In [ ]:
computeStatisticsHistograms(
    IMG, geometry=AOI_BBOX, geometry_type="esriGeometryEnvelope",
    mosaic_rule=YEAR_2020,
    pixel_size={"x": 90, "y": 90, "spatialReference": {"wkid": 4326}},  # 90 degrees, not 90 m!
)

## 8. `identifyPixelValue` — identify the pixel value at a point

A single x/y only — the underlying `identify` operation silently collapses other geometry types to their centroid rather than sampling an area.

In [ ]:
identifyPixelValue(IMG, x=-118.55, y=36.15, mosaic_rule=YEAR_2020)

`return_catalog_items=True` also returns the footprint attributes of the raster(s) contributing at this point (e.g. which mosaic source/year):

In [ ]:
r = identifyPixelValue(IMG, x=-118.55, y=36.15, mosaic_rule=YEAR_2020, return_catalog_items=True)
r["value"], r["catalog_items"][0]["attributes"]

## 9. `getSamples` — sample pixel values at multiple points

One batch request for every point, with a per-point `identify` fallback (see the function's docstring) for any point the batch call can't resolve.

In [ ]:
points = [(-118.55, 36.15), (-118.60, 36.10), (-118.50, 36.20)]
getSamples(IMG, points, mosaic_rule=YEAR_2020)

## 10. `queryBoundary` — the true coverage boundary

Unlike the rectangular `extent` from `getLayerInfo`, this is the actual coverage shape — useful as a pre-flight check for whether a service covers an AOI at all before spending an `exportImage`/`computeStatisticsHistograms` call on it.

In [ ]:
boundary = queryBoundary(IMG)
print(boundary["type"], "- area:", boundary["area"])
print(len(boundary["coordinates"][0]), "vertices in the outer ring")